In [1]:
!pip install -U transformers datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 71.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 89.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 112.6 MB/s eta 0:00:0000:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: datasets
    Found existing installatio

In [2]:
# ------------------------------------------------------------
# 0) Install (run once if needed)
# ------------------------------------------------------------
# !pip install -U transformers datasets scikit-learn

# ------------------------------------------------------------
# 1) Imports
# ------------------------------------------------------------
from transformers import AutoTokenizer, AutoModel
from transformers import TrainingArguments, Trainer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

# ------------------------------------------------------------
# 2) Load Data
# ------------------------------------------------------------
df = pd.read_csv("/kaggle/input/datasets/shuvo128/newbuc/IMDB_Cleaned (1).csv")

text_column = "cleaned_review" if "cleaned_review" in df.columns else "review"
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

texts  = df[text_column].astype(str).tolist()
labels = (df["sentiment"] == "positive").astype(int).tolist()

# ------------------------------------------------------------
# 3) FIXED SPLIT (40k / 5k / 5k)
# ------------------------------------------------------------
train_texts = texts[:40000]
train_labels = labels[:40000]

val_texts = texts[40000:45000]
val_labels = labels[40000:45000]

test_texts = texts[45000:50000]
test_labels = labels[45000:50000]

print(f"Train: {len(train_texts)}, Val: {len(val_texts)}, Test: {len(test_texts)}")

# ------------------------------------------------------------
# 4) TF-IDF
# ------------------------------------------------------------
tfidf = TfidfVectorizer(max_features=5000)

tfidf.fit(train_texts)

train_tfidf = tfidf.transform(train_texts).toarray()
val_tfidf   = tfidf.transform(val_texts).toarray()
test_tfidf  = tfidf.transform(test_texts).toarray()

# ------------------------------------------------------------
# 5) ELECTRA Tokenizer
# ------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("google/electra-base-discriminator")

def tokenize(texts):
    return tokenizer(texts, truncation=True, padding=True, max_length=256)

train_enc = tokenize(train_texts)
val_enc   = tokenize(val_texts)
test_enc  = tokenize(test_texts)

# ------------------------------------------------------------
# 6) Dataset Class (NOW includes TF-IDF)
# ------------------------------------------------------------
class HybridDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, tfidf_features, labels):
        self.encodings = encodings
        self.tfidf = tfidf_features
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["tfidf"] = torch.tensor(self.tfidf[idx], dtype=torch.float)
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = HybridDataset(train_enc, train_tfidf, train_labels)
val_dataset   = HybridDataset(val_enc, val_tfidf, val_labels)
test_dataset  = HybridDataset(test_enc, test_tfidf, test_labels)

# ------------------------------------------------------------
# 7) HYBRID MODEL (ELECTRA + TF-IDF)
# ------------------------------------------------------------
class ElectraTFIDF(nn.Module):
    def __init__(self, model_name, tfidf_dim, num_labels):
        super().__init__()
        self.electra = AutoModel.from_pretrained(model_name)
        hidden_size = self.electra.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + tfidf_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_labels)
        )

    def forward(self, input_ids, attention_mask, tfidf, labels=None):
        outputs = self.electra(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token

        combined = torch.cat((cls_output, tfidf), dim=1)

        logits = self.classifier(combined)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

# Initialize model
model = ElectraTFIDF(
    model_name="google/electra-base-discriminator",
    tfidf_dim=5000,
    num_labels=2
)

# ------------------------------------------------------------
# 8) Metrics
# ------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# ------------------------------------------------------------
# 9) Training Arguments
# ------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_steps=100,
    save_strategy="no",
    report_to="none"
)

# ------------------------------------------------------------
# 10) Trainer
# ------------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# ------------------------------------------------------------
# 11) Train
# ------------------------------------------------------------
trainer.train()

# ------------------------------------------------------------
# 12) Validation
# ------------------------------------------------------------
print("\nValidation Results:")
val_results = trainer.evaluate()
print(val_results)

# ------------------------------------------------------------
# 13) Test
# ------------------------------------------------------------
print("\nTest Results:")
test_results = trainer.predict(test_dataset)

test_preds = np.argmax(test_results.predictions, axis=1)

precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average='binary'
)
acc = accuracy_score(test_labels, test_preds)

print({
    "accuracy": acc,
    "precision": precision,
    "recall": recall,
    "f1": f1
})

Train: 40000, Val: 5000, Test: 5000


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...


Validation Results:


Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
0.065985,0.286044,7500,0.940600,0.931512,0.949939,0.940636


{'eval_loss': 0.28604400157928467, 'eval_accuracy': 0.9406, 'eval_precision': 0.9315122723673792, 'eval_recall': 0.9499394428744449, 'eval_f1': 0.9406356186288227}

Test Results:


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'accuracy': 0.9482, 'precision': 0.9448818897637795, 'recall': 0.9527590313616514, 'f1': 0.9488041114844831}
